In [15]:
import re
import pandas as pd
import tdt
import os

In [2]:
s = """%dir1='JZ_exp-220620-114107--A2A-DLS-ctrl3-halo6(context)';
%stamp1=[37;38];
%stamp2=[195;196]; 
%stamp3=[250;251];


%dir1='JZ_exp-220924-113657-A2A-DLS-ctrl1-halo6(context)';
%stamp1=[25;26];
%stamp2=[59;60]; 
%stamp3=[146;147];

%dir1='JZ_exp-220924-114527-A2A-DLS-ctrl2-halo6(context)';
%stamp1=[28;29];
%stamp2=[68;69]; 
%stamp3=[238;239];

%dir1='JZ_exp-221008-121735-A2A-DLS-ctrl1-halo6(context)';
%stamp1=[52;53];
%stamp2=[72;73]; 
%stamp3=[116;117];

%dir1='JZ_exp-221008-122925-A2A-DLS-ctrl2-halo6(context)';
%stamp1=[64;65];
%stamp2=[157;158]; 
%stamp3=[192;193];

%dir1='JZ_exp-220621-113014-A2A-DLS-ctrl1-halo7';
%stamp1=[197;217];
%stamp2=[238;262]; 
%stamp3=[335;372];

%dir1='JZ_exp-220621-113951-A2A-DLS-ctrl2-halo7';
%stamp1=[49;111];
%stamp2=[219;243]; 
%stamp3=[343;368];
 
%dir1='JZ_exp-220621-115000-A2A-DLS-ctrl3-halo7';
%stamp1=[26;37];
%stamp2=[113;123]; 
%stamp3=[140;149];

%dir1='JZ_exp-220925-114224-A2A-DLS-ctrl1-halo7';
%stamp1=[224;245];
%stamp2=[259;294]; 
%stamp3=[432;453];

%dir1='JZ_exp-220925-115311-A2A-DLS-ctrl2-halo7';
%stamp1=[71;108];
%stamp2=[138;184]; 
%stamp3=[462;483];

%dir1='JZ_exp-221009-153547-A2A-DLS-ctrl1-halo7';
%stamp1=[150;224];
%stamp2=[338;364]; 
%stamp3=[467;521];

%dir1='JZ_exp-221009-155425-A2A-DLS-ctrl2-halo7';
%stamp1=[51;115];
%stamp2=[150;194]; 
%stamp3=[222;256];

%dir1='JZ_exp-220622-114003-A2A-DLS-ctrl1-halo8(low)';
%stamp1=[40;81];
%stamp2=[197;231]; 
%stamp3=[248;286];

%dir1='JZ_exp-220622-115038-A2A-DLS-ctrl2-halo8(low)';
%stamp1=[84;104];
%stamp2=[166;180]; 
%stamp3=[301;314];

%dir1='JZ_exp-220622-120017-A2A-DLS-ctrl3-halo8(low)';
%stamp1=[21;22];
%stamp2=[164;204]; 
%stamp3=[236;251];

%dir1='JZ_exp-220926-114825-A2A-DLS-ctrl1-halo8(low)';
%stamp1=[188;226];
%stamp2=[265;313]; 
%stamp3=[492;559];


%dir1='JZ_exp-220926-120050-A2A-DLS-ctrl2-halo8(low)';
%stamp1=[47;78];
%stamp2=[150;173]; 
%stamp3=[327;352];

%dir1='JZ_exp-221010-111808-A2A-DLS-ctrl1-halo8(low)';
%stamp1=[39;101];
%stamp2=[145;167]; 
%stamp3=[306;438];
 
% dir1='JZ_exp-221010-112921-A2A-DLS-ctrl2-halo8(low)';
% stamp1=[83;131];
% stamp2=[169;199]; 
% stamp3=[209;238];"""

In [11]:
import re

# Create a list to store the parsed data
parsed_data = []

# Split the string into lines and process each group
lines = s.strip().split('\n')

current_entry = {}
for line in lines:
    line = line.strip()
    if not line:  # Skip empty lines
        continue

    # Check if line starts with % (some lines might have % commented out)
    if not line.startswith('%'):
        continue

    # Remove the % symbol
    line = line.lstrip('%')

    # Parse directory
    if line.startswith('dir1='):
        # If we have a previous entry, save it
        if current_entry and len(current_entry) == 4:  # dir + 3 stamps
            parsed_data.append(current_entry)
        # Start new entry
        current_entry = {}
        # Extract directory name (removing quotes)
        dir_match = re.match(r"dir1='([^']*)'", line)
        if dir_match:
            current_entry['directory'] = dir_match.group(1)

    # Parse stamps
    elif line.startswith(('stamp1=', 'stamp2=', 'stamp3=')):
        # Extract stamp number and values
        stamp_match = re.match(r"stamp(\d+)=\[(\d+);(\d+)\]", line)
        if stamp_match:
            stamp_num = int(stamp_match.group(1))
            start = int(stamp_match.group(2))
            end = int(stamp_match.group(3))
            current_entry[f'stamp{stamp_num}'] = (start, end)

# Don't forget to add the last entry
if current_entry and len(current_entry) == 4:
    parsed_data.append(current_entry)

# Print the first few entries to verify
for entry in parsed_data:
    print(entry)

{'directory': 'JZ_exp-220620-114107--A2A-DLS-ctrl3-halo6(context)', 'stamp1': (37, 38), 'stamp2': (195, 196), 'stamp3': (250, 251)}
{'directory': 'JZ_exp-220924-113657-A2A-DLS-ctrl1-halo6(context)', 'stamp1': (25, 26), 'stamp2': (59, 60), 'stamp3': (146, 147)}
{'directory': 'JZ_exp-220924-114527-A2A-DLS-ctrl2-halo6(context)', 'stamp1': (28, 29), 'stamp2': (68, 69), 'stamp3': (238, 239)}
{'directory': 'JZ_exp-221008-121735-A2A-DLS-ctrl1-halo6(context)', 'stamp1': (52, 53), 'stamp2': (72, 73), 'stamp3': (116, 117)}
{'directory': 'JZ_exp-221008-122925-A2A-DLS-ctrl2-halo6(context)', 'stamp1': (64, 65), 'stamp2': (157, 158), 'stamp3': (192, 193)}
{'directory': 'JZ_exp-220621-113014-A2A-DLS-ctrl1-halo7', 'stamp1': (197, 217), 'stamp2': (238, 262), 'stamp3': (335, 372)}
{'directory': 'JZ_exp-220621-113951-A2A-DLS-ctrl2-halo7', 'stamp1': (49, 111), 'stamp2': (219, 243), 'stamp3': (343, 368)}
{'directory': 'JZ_exp-220621-115000-A2A-DLS-ctrl3-halo7', 'stamp1': (26, 37), 'stamp2': (113, 123), 'st

In [17]:
os.listdir('/Volumes/Seagate-HFS+/MATLAB/')

PermissionError: [Errno 1] Operation not permitted: '/Volumes/Seagate-HFS+/MATLAB/'

In [14]:
data = tdt.read_block('/Volumes/Seagate-HFS+/MATLAB/JZ_exp-221010-111808-A2A-DLS-ctrl1-halo8(low)')

/Users/xinding/Insync/bac2qh@gmail.com/Google Drive/work_and_life/SZY/.venv/lib/python3.13/site-packages/tdt/__init__.py:51: SyntaxWarning: invalid escape sequence '\W'
  fixed_name = re.sub('\W|^(?=\d)', '_', var_str)


PermissionError: [Errno 1] Operation not permitted: '/Volumes/Seagate-HFS+/MATLAB/JZ_exp-221010-111808-A2A-DLS-ctrl1-halo8(low)/'